# SIC k9 — Independence number

$SIC$ k=9, d=3, $\theta$-37.8, $\alpha$=36

In [ ]:
!pip install pulp

In [ ]:
import numpy as np
import itertools
import networkx as nx
import pulp

def generate_rays(k):
    if k < 1 or int(k) != k:
        raise ValueError("k must be a positive integer")
    k = int(k)
    q = np.exp(2j * np.pi / k)

    vectors, labels = [], []
    vectors += [
        np.array([1, 0, 0], dtype=complex),
        np.array([0, 1, 0], dtype=complex),
        np.array([0, 0, 1], dtype=complex),
    ]
    labels += ["I:(1,0,0)", "I:(0,1,0)", "I:(0,0,1)"]

    for i in range(1,k + 1):
        qi = q ** i
        vectors += [
            np.array([1, -qi, 0], dtype=complex),
            np.array([1, 0, -qi], dtype=complex),
            np.array([0, 1, -qi], dtype=complex),
        ]
        labels += [f"II:i={i}:(1,-q^i,0)",
                   f"II:i={i}:(1,0,-q^i)",
                   f"II:i={i}:(0,1,-q^i)"]
    for i in range(1, k + 1):
        for j in range(1, k + 1):
            qi, qj = q ** i, q ** j
            vectors.append(np.array([1, qi, qj], dtype=complex))
            labels.append(f"III:(i={i},j={j}):(1,q^i,q^j)")

    expected_n = 3 + 3 * k + k * k
    assert len(vectors) == expected_n, \
        f"generated {len(vectors)} rays, expected {expected_n}"

    return vectors, labels

def is_orthogonal(u, v, tol=1e-7):
    nu = u / np.linalg.norm(u)
    nv = v / np.linalg.norm(v)
    return abs(np.vdot(nu, nv)) < tol


def is_same_ray(u, v, tol=1e-7):
    nu = u / np.linalg.norm(u)
    nv = v / np.linalg.norm(v)
    # rays coincide iff |<u|v>| ~ 1
    return abs(abs(np.vdot(nu, nv)) - 1.0) < tol

def build_orthogonality_graph(vectors, tol=1e-7, verbose=True):
    n = len(vectors)
    G = nx.Graph()
    G.add_nodes_from(range(n))

    duplicate_pairs = []
    for i, j in itertools.combinations(range(n), 2):
        if is_same_ray(vectors[i], vectors[j], tol):
            duplicate_pairs.append((i, j))
        if is_orthogonal(vectors[i], vectors[j], tol):
            G.add_edge(i, j)

    if verbose:
        if duplicate_pairs:
            print(f"WARNING: {len(duplicate_pairs)} duplicate ray pair(s) "
                  f"found (same ray up to phase): {duplicate_pairs[:10]}"
                  f"{' ...' if len(duplicate_pairs) > 10 else ''}")
        else:
            print("Sanity check: no duplicate rays found.")

    return G

def independence_number(G, verbose=True):
    n = G.number_of_nodes()
    prob = pulp.LpProblem("max_independent_set", pulp.LpMaximize)
    x = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in G.nodes()}

    prob += pulp.lpSum(x[i] for i in G.nodes())
    for i, j in G.edges():
        prob += x[i] + x[j] <= 1

    status = prob.solve(pulp.PULP_CBC_CMD(msg=False))
    if pulp.LpStatus[status] != "Optimal":
        raise RuntimeError(f"ILP did not solve to optimality: "
                            f"{pulp.LpStatus[status]}")

    alpha = int(round(pulp.value(prob.objective)))
    chosen = sorted(i for i in G.nodes() if x[i].value() > 0.5)

    for i, j in itertools.combinations(chosen, 2):
        assert not G.has_edge(i, j), \
            f"ILP result invalid: {i} and {j} are adjacent but both chosen"

    return alpha, chosen


def alpha_formula(k):
    if k % 3 == 0:
        return k * k // 3 + k
    else:
        return 1 + k * k


if __name__ == "__main__":
    k=9

    vectors, labels = generate_rays(k)
    n = len(vectors)
    d = 3  # qutrit

    print(f"k = {k}")
    print(f"n (number of rays) = {n}  "
          f"(expected 3 + 3*{k} + {k}^2 = {3 + 3*k + k*k})")

    G = build_orthogonality_graph(vectors, tol=1e-7, verbose=True)
    print(f"Orthogonality graph: {G.number_of_nodes()} vertices, "
          f"{G.number_of_edges()} edges")

    alpha, chosen = independence_number(G, verbose=True)

    print(f"\nIndependence number alpha(G) = {alpha}")
    print(f"alpha * d = {alpha} * {d} = {alpha * d}")
    print(f"n/3 = {n/3:.4f}   (paper's SIC criterion: alpha < n/3 => SIC)")
    print(f"SIC (per RH-type inequality)? {'YES' if alpha < n / 3 else 'NO'}")

    expected_alpha = alpha_formula(k)
    print(f"\nClosed-form check (paper Eq. 8): alpha_formula({k}) = "
          f"{expected_alpha}")
    print("MATCH" if alpha == expected_alpha else
          f"MISMATCH (computed {alpha}, formula gives {expected_alpha})")

    print("\nOne maximum independent set (0-based indices, with labels):")
    for idx in chosen:
        print(f"  {idx}: {labels[idx]}")

k = 9
n (number of rays) = 111  (expected 3 + 3*9 + 9^2 = 111)
Sanity check: no duplicate rays found.
Orthogonality graph: 111 vertices, 354 edges

Independence number alpha(G) = 36
alpha * d = 36 * 3 = 108
n/3 = 37.0000   (paper's SIC criterion: alpha < n/3 => SIC)
SIC (per RH-type inequality)? YES

Closed-form check (paper Eq. 8): alpha_formula(9) = 36
MATCH

One maximum independent set (0-based indices, with labels):
  5: II:i=1:(0,1,-q^i)
  6: II:i=2:(1,-q^i,0)
  7: II:i=2:(1,0,-q^i)
  12: II:i=4:(1,-q^i,0)
  20: II:i=6:(0,1,-q^i)
  22: II:i=7:(1,0,-q^i)
  26: II:i=8:(0,1,-q^i)
  27: II:i=9:(1,-q^i,0)
  28: II:i=9:(1,0,-q^i)
  30: III:(i=1,j=1):(1,q^i,q^j)
  32: III:(i=1,j=3):(1,q^i,q^j)
  33: III:(i=1,j=4):(1,q^i,q^j)
  34: III:(i=1,j=5):(1,q^i,q^j)
  35: III:(i=1,j=6):(1,q^i,q^j)
  37: III:(i=1,j=8):(1,q^i,q^j)
  48: III:(i=3,j=1):(1,q^i,q^j)
  50: III:(i=3,j=3):(1,q^i,q^j)
  52: III:(i=3,j=5):(1,q^i,q^j)
  53: III:(i=3,j=6):(1,q^i,q^j)
  55: III:(i=3,j=8):(1,q^i,q^j)
  66: III:(